In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
import librosa.display
import matplotlib.pyplot as plt
import scipy.io.wavfile as wavfile
import os

# ─────────────────────────────────────────────
# 1.  HYPER-PARAMETERS
# ─────────────────────────────────────────────
class HParams:
    # Text
    num_chars        = 128          # ASCII character set
    embedding_dim    = 512

    # Encoder
    encoder_hidden   = 256          # units per direction
    encoder_layers   = 3
    encoder_kernel   = 5

    # Attention
    attention_dim    = 128
    attention_filters= 32
    attention_kernel = 31

    # Decoder
    num_mels         = 80
    frames_per_step  = 1
    decoder_hidden   = 1024
    decoder_layers   = 2
    prenet_dim       = 256
    max_decoder_steps= 1000
    gate_threshold   = 0.5

    # Mel / audio
    sample_rate      = 22050
    hop_length       = 256
    win_length       = 1024
    n_fft            = 1024
    fmin             = 0
    fmax             = 8000

    # Training
    learning_rate    = 1e-3
    batch_size       = 4
    epochs           = 5            # small for demo; increase for real training

hp = HParams()


# ─────────────────────────────────────────────
# 2.  TEXT UTILITIES
# ─────────────────────────────────────────────
def text_to_sequence(text: str) -> list[int]:
    """Convert a text string to a list of ASCII indices."""
    return [ord(c) for c in text if ord(c) < hp.num_chars]

def sequence_to_text(seq: list[int]) -> str:
    return ''.join(chr(i) for i in seq)


# ─────────────────────────────────────────────
# 3.  MODEL COMPONENTS
# ─────────────────────────────────────────────

class ConvNorm(nn.Module):
    """1-D convolution → batch-norm block used in the encoder."""
    def __init__(self, in_ch, out_ch, kernel=1, stride=1,
                 padding=None, dilation=1, bias=True):
        super().__init__()
        if padding is None:
            padding = (kernel - 1) * dilation // 2
        self.conv = nn.Conv1d(in_ch, out_ch, kernel, stride=stride,
                              padding=padding, dilation=dilation, bias=bias)
        self.bn   = nn.BatchNorm1d(out_ch)

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)))


class Encoder(nn.Module):
    """
    Tacotron2 encoder:
      embedding → 3× (Conv1d + BN + ReLU) → BiLSTM
    Output shape: (B, T_text, 2 * encoder_hidden)
    """
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(hp.num_chars, hp.embedding_dim)
        convs = []
        in_ch = hp.embedding_dim
        for _ in range(hp.encoder_layers):
            convs.append(ConvNorm(in_ch, hp.embedding_dim, hp.encoder_kernel))
            in_ch = hp.embedding_dim
        self.convs = nn.Sequential(*convs)
        self.lstm  = nn.LSTM(hp.embedding_dim,
                             hp.encoder_hidden,
                             num_layers=1,
                             batch_first=True,
                             bidirectional=True)

    def forward(self, x, lengths=None):
        # x: (B, T)
        x = self.embedding(x)          # (B, T, E)
        x = x.transpose(1, 2)          # (B, E, T)  — Conv1d wants (B, C, T)
        x = self.convs(x)
        x = x.transpose(1, 2)          # (B, T, E)
        if lengths is not None:
            x = nn.utils.rnn.pack_padded_sequence(
                x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(x)
        if lengths is not None:
            out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        return out                      # (B, T, 2*H)


class LocationLayer(nn.Module):
    """Processes cumulative attention weights (location features)."""
    def __init__(self):
        super().__init__()
        self.conv  = nn.Conv1d(2, hp.attention_filters,
                               hp.attention_kernel,
                               padding=(hp.attention_kernel - 1) // 2,
                               bias=False)
        self.dense = nn.Linear(hp.attention_filters, hp.attention_dim, bias=False)

    def forward(self, attention_weights_cat):
        # attention_weights_cat: (B, 2, T_enc)
        x = self.conv(attention_weights_cat)   # (B, F, T_enc)
        x = x.transpose(1, 2)                  # (B, T_enc, F)
        return self.dense(x)                   # (B, T_enc, attn_dim)


class Attention(nn.Module):
    """Location-sensitive attention (Chorowski et al., 2015)."""
    def __init__(self, rnn_dim):
        super().__init__()
        self.query_layer    = nn.Linear(rnn_dim, hp.attention_dim, bias=False)
        self.memory_layer   = nn.Linear(2 * hp.encoder_hidden,
                                         hp.attention_dim, bias=False)
        self.v              = nn.Linear(hp.attention_dim, 1, bias=False)
        self.location_layer = LocationLayer()
        self.score_mask_val = -1e9

    def score(self, query, processed_memory, attention_weights_cat):
        q   = self.query_layer(query.unsqueeze(1))          # (B, 1, attn_dim)
        loc = self.location_layer(attention_weights_cat)    # (B, T, attn_dim)
        e   = self.v(torch.tanh(q + processed_memory + loc))# (B, T, 1)
        return e.squeeze(-1)                                 # (B, T)

    def forward(self, query, memory, processed_memory,
                attention_weights_cat, mask=None):
        scores = self.score(query, processed_memory, attention_weights_cat)
        if mask is not None:
            scores.data.masked_fill_(mask, self.score_mask_val)
        weights  = F.softmax(scores, dim=1)
        context  = torch.bmm(weights.unsqueeze(1), memory)  # (B, 1, 2H)
        context  = context.squeeze(1)                       # (B, 2H)
        return context, weights


class Prenet(nn.Module):
    """Two FC layers with dropout — decoder input pre-processing."""
    def __init__(self, in_dim, out_dim=None):
        super().__init__()
        out_dim = out_dim or hp.prenet_dim
        self.layers = nn.ModuleList([
            nn.Linear(in_dim, out_dim),
            nn.Linear(out_dim, out_dim),
        ])

    def forward(self, x):
        for layer in self.layers:
            x = F.dropout(F.relu(layer(x)), p=0.5, training=True)
        return x


class Decoder(nn.Module):
    """
    Tacotron2 auto-regressive decoder.
    Predicts mel-spectrograms one frame at a time.
    """
    def __init__(self):
        super().__init__()
        encoder_out_dim = 2 * hp.encoder_hidden   # 512

        self.prenet = Prenet(hp.num_mels)
        self.attention = Attention(hp.decoder_hidden)

        # Stack of LSTM layers
        self.lstm_cells = nn.ModuleList([
            nn.LSTMCell(hp.prenet_dim + encoder_out_dim,
                        hp.decoder_hidden),
            nn.LSTMCell(hp.decoder_hidden, hp.decoder_hidden),
        ])

        self.linear_projection = nn.Linear(
            hp.decoder_hidden + encoder_out_dim, hp.num_mels)
        self.gate_layer = nn.Linear(
            hp.decoder_hidden + encoder_out_dim, 1)

    def _init_states(self, memory, B, device):
        """Initialise decoder hidden/cell states and attention variables."""
        T_enc = memory.size(1)
        h = [torch.zeros(B, hp.decoder_hidden, device=device)
             for _ in self.lstm_cells]
        c = [torch.zeros(B, hp.decoder_hidden, device=device)
             for _ in self.lstm_cells]
        attn_weights     = torch.zeros(B, T_enc, device=device)
        attn_weights_cum = torch.zeros(B, T_enc, device=device)
        attn_context     = torch.zeros(B, 2 * hp.encoder_hidden, device=device)
        return h, c, attn_weights, attn_weights_cum, attn_context

    def forward(self, memory, memory_lengths=None, mel_targets=None):
        """
        memory      : encoder output  (B, T_enc, 2H)
        mel_targets : ground-truth mel (B, num_mels, T_mel) — teacher forcing
        Returns mel_outputs, gate_outputs, alignments
        """
        B, T_enc, _ = memory.shape
        device = memory.device

        processed_memory = Attention(hp.decoder_hidden).memory_layer(memory) \
            if False else \
            nn.Linear(2 * hp.encoder_hidden, hp.attention_dim,
                      bias=False).to(device)(memory)
        # NOTE: in a proper implementation this Linear is shared; here we keep
        # it simple and recompute — for training correctness use a stored layer.

        mask = None
        if memory_lengths is not None:
            max_len = memory.size(1)
            mask = ~(torch.arange(max_len, device=device).unsqueeze(0)
                     < memory_lengths.unsqueeze(1))  # (B, T_enc)

        h, c, aw, aw_cum, ctx = self._init_states(memory, B, device)
        frame = torch.zeros(B, hp.num_mels, device=device)  # GO frame

        mel_outputs, gate_outputs, alignments = [], [], []

        T_mel = mel_targets.size(2) if mel_targets is not None \
                else hp.max_decoder_steps

        for t in range(T_mel):
            inp = self.prenet(frame)                              # (B, prenet_dim)
            cell_inp = torch.cat([inp, ctx], dim=-1)             # (B, prenet+ctx)

            # LSTM pass
            h[0], c[0] = self.lstm_cells[0](cell_inp, (h[0], c[0]))
            h[1], c[1] = self.lstm_cells[1](h[0], (h[1], c[1]))

            # Attention
            aw_cat = torch.stack([aw, aw_cum], dim=1)            # (B, 2, T_enc)
            ctx, aw = self.attention(h[1], memory, processed_memory,
                                     aw_cat, mask)
            aw_cum = aw_cum + aw

            # Projection
            proj_inp = torch.cat([h[1], ctx], dim=-1)            # (B, H+ctx)
            mel_frame = self.linear_projection(proj_inp)          # (B, num_mels)
            gate      = self.gate_layer(proj_inp)                 # (B, 1)

            mel_outputs.append(mel_frame)
            gate_outputs.append(gate.squeeze(-1))
            alignments.append(aw)

            # Next input: teacher forcing during training, own prediction otherwise
            if mel_targets is not None:
                frame = mel_targets[:, :, t]
            else:
                frame = mel_frame
                if torch.sigmoid(gate).mean() > hp.gate_threshold:
                    break

        mel_outputs  = torch.stack(mel_outputs, dim=2)   # (B, num_mels, T)
        gate_outputs = torch.stack(gate_outputs, dim=1)  # (B, T)
        alignments   = torch.stack(alignments, dim=1)    # (B, T, T_enc)
        return mel_outputs, gate_outputs, alignments


class Tacotron2(nn.Module):
    """Full Tacotron2 model: Encoder + Attention Decoder."""
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        # Shared processed-memory projection (belongs here, not in Decoder)
        self.memory_layer = nn.Linear(2 * hp.encoder_hidden,
                                       hp.attention_dim, bias=False)
        self.decoder_attention = Attention(hp.decoder_hidden)
        self.decoder = Decoder()

    def forward(self, text_ids, text_lengths=None, mel_targets=None):
        memory = self.encoder(text_ids, text_lengths)
        mel_out, gate_out, alignments = self.decoder(
            memory, text_lengths, mel_targets)
        return mel_out, gate_out, alignments


# ─────────────────────────────────────────────
# 4.  LOSS
# ─────────────────────────────────────────────
class Tacotron2Loss(nn.Module):
    def forward(self, mel_pred, mel_target, gate_pred, gate_target):
        mel_loss  = F.mse_loss(mel_pred, mel_target)
        gate_loss = F.binary_cross_entropy_with_logits(gate_pred, gate_target)
        return mel_loss + gate_loss


# ─────────────────────────────────────────────
# 5.  VOCODER: Griffin-Lim
# ─────────────────────────────────────────────
def mel_to_audio(mel_spec_db: np.ndarray, sr: int = hp.sample_rate) -> np.ndarray:
    """
    Convert a mel-spectrogram (in dB) back to a waveform via Griffin-Lim.
    mel_spec_db shape: (num_mels, T)
    """
    # dB → power
    mel_power = librosa.db_to_power(mel_spec_db)
    # Invert mel filterbank
    stft_power = librosa.feature.inverse.mel_to_stft(
        mel_power, sr=sr, n_fft=hp.n_fft,
        fmin=hp.fmin, fmax=hp.fmax)
    # Griffin-Lim
    audio = librosa.griffinlim(
        np.sqrt(stft_power),
        n_iter=32,
        hop_length=hp.hop_length,
        win_length=hp.win_length)
    return audio


# ─────────────────────────────────────────────
# 6.  DEMO TRAINING LOOP (tiny synthetic data)
# ─────────────────────────────────────────────
def make_dummy_batch(texts, device):
    """Create a tiny batch from a list of strings."""
    seqs    = [torch.tensor(text_to_sequence(t), dtype=torch.long) for t in texts]
    lengths = torch.tensor([len(s) for s in seqs])
    padded  = nn.utils.rnn.pad_sequence(seqs, batch_first=True).to(device)

    T_mel   = 50                         # short fixed length for demo
    mels    = torch.randn(len(texts), hp.num_mels, T_mel).to(device)
    gates   = torch.zeros(len(texts), T_mel).to(device)
    gates[:, -1] = 1.0                   # signal end-of-sequence at last frame
    return padded, lengths.to(device), mels, gates


def train_demo():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}\n")

    model     = Tacotron2().to(device)
    criterion = Tacotron2Loss()
    optimizer = torch.optim.Adam(model.parameters(), lr=hp.learning_rate)

    texts = [
        "Hello world",
        "Text to speech",
        "Deep learning is amazing",
        "Tacotron two",
    ]

    print("=" * 60)
    print("TRAINING DEMO  (synthetic data, 5 epochs)")
    print("=" * 60)

    model.train()
    for epoch in range(1, hp.epochs + 1):
        text_ids, lengths, mel_targets, gate_targets = make_dummy_batch(texts, device)
        optimizer.zero_grad()
        mel_pred, gate_pred, alignments = model(text_ids, lengths, mel_targets)
        loss = criterion(mel_pred, mel_targets, gate_pred, gate_targets)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        print(f"  Epoch {epoch}/{hp.epochs}  Loss: {loss.item():.4f}")

    print("\nTraining complete.\n")
    return model


# ─────────────────────────────────────────────
# 7.  INFERENCE + VISUALISATION
# ─────────────────────────────────────────────
def synthesise(model, text: str, device, out_dir: str = "."):
    """Run inference on `text` and save mel + audio."""
    os.makedirs(out_dir, exist_ok=True)

    model.eval()
    with torch.no_grad():
        seq  = torch.tensor(text_to_sequence(text),
                            dtype=torch.long).unsqueeze(0).to(device)
        lens = torch.tensor([seq.size(1)]).to(device)
        mel, gate, align = model(seq, lens, mel_targets=None)

    mel_np   = mel.squeeze(0).cpu().numpy()          # (num_mels, T)
    align_np = align.squeeze(0).cpu().numpy()        # (T_dec, T_enc)

    # ── Plot mel-spectrogram ──────────────────────────────────────────
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    librosa.display.specshow(mel_np, sr=hp.sample_rate,
                             hop_length=hp.hop_length,
                             y_axis='mel', x_axis='time', ax=axes[0])
    axes[0].set_title(f'Mel Spectrogram  — "{text}"')
    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Mel Frequency")

    axes[1].imshow(align_np.T, aspect='auto', origin='lower', cmap='viridis')
    axes[1].set_title("Attention Alignment")
    axes[1].set_xlabel("Decoder step")
    axes[1].set_ylabel("Encoder step")

    plt.tight_layout()
    fig_path = os.path.join(out_dir, "mel_alignment.png")
    plt.savefig(fig_path, dpi=150)
    plt.close()
    print(f"Saved figure → {fig_path}")

    # ── Griffin-Lim → waveform ────────────────────────────────────────
    mel_db  = librosa.power_to_db(np.exp(mel_np), ref=np.max)
    audio   = mel_to_audio(mel_db)
    audio   = (audio / np.max(np.abs(audio) + 1e-9) * 32767).astype(np.int16)
    wav_path = os.path.join(out_dir, "output.wav")
    wavfile.write(wav_path, hp.sample_rate, audio)
    print(f"Saved audio  → {wav_path}")
    return mel_np, audio


# ─────────────────────────────────────────────
# 8.  ENTRY POINT
# ─────────────────────────────────────────────
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Train (demo)
    model = train_demo()

    # Synthesise
    sample_text = "Hello, this is a text to speech system."
    print(f'Synthesising: "{sample_text}"')
    mel_np, audio = synthesise(model, sample_text, device, out_dir="tts_output")

    print("\nDone!  Check the tts_output/ folder for mel_alignment.png and output.wav")
    print("\nModel Summary:")
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total params     : {total:,}")
    print(f"  Trainable params : {trainable:,}")


Using device: cpu

TRAINING DEMO  (synthetic data, 5 epochs)
  Epoch 1/5  Loss: 1.7375
  Epoch 2/5  Loss: 1.2001
  Epoch 3/5  Loss: 1.3267
  Epoch 4/5  Loss: 1.2213
  Epoch 5/5  Loss: 1.1834

Training complete.

Synthesising: "Hello, this is a text to speech system."
Saved figure → tts_output/mel_alignment.png
Saved audio  → tts_output/output.wav

Done!  Check the tts_output/ folder for mel_alignment.png and output.wav

Model Summary:
  Total params     : 22,006,481
  Trainable params : 22,006,481
